# HW2-1 单模型版本

Kaggle: `ml2021spring-hw2` · **public 0.74719 / private 0.74657**

配方（每一条都是实验换来的，理由见 `README.md`）：

| 组件 | 选择 | 为什么 |
|---|---|---|
| 输入 | k=2 窗口，5×429 = 2145 维 | 数据是 10 ms 帧，拼邻帧能补上下文；k=2 比 k=1/k=3 都好 |
| 架构 | `2145→1024→512→128→39`，Linear+BN+ReLU+Dropout 0.2 | 2.8 M 参数是容量拐点，再大就背题（v5 到 12.8 M 时 train 0.997） |
| 优化器 | AdamW lr 1.5e-3 + OneCycle，30 epoch | BN 让 lr 能上 1e-3 量级；OneCycle 省掉手调 epoch |
| 数据 | **100%**，不切验证集 | epoch 数已在上一步确定，留 20% 不用等于白扔 25 万帧 |
| 推理 | TTA：窗口偏移 −1/0/+1 取平均 | 边界帧的上下文有偏，平移能抹平一部分 |

跑完输出 `submission.csv` 和 `probs.npy`（后者给集成用）。


## Preparing Data

> 官方 notebook 这一步是从 Google Drive 下载 `timit_11/*.npy`；这里改成从比赛数据挂载点读取，其余不变。

In [ ]:
import os, glob, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
DATA_DIR = os.path.dirname(glob.glob('/kaggle/input/**/train_11.npy', recursive=True)[0])
SUB_PATH = glob.glob('/kaggle/input/**/sampleSubmission.csv', recursive=True)[0]

lab = np.load(os.path.join(DATA_DIR, 'train_label_11.npy'))
if lab.dtype.kind in 'US':          # 这份数据的标签存成了字符串 '0'..'38'
    lab = lab.astype(np.int64)
train = np.load(os.path.join(DATA_DIR, 'train_11.npy'), mmap_mode='r')   # float64, 4.2 GB -> 用 mmap
test = np.load(os.path.join(DATA_DIR, 'test_11.npy'), mmap_mode='r')
print('train', train.shape, train.dtype, '| test', test.shape, '| labels', lab.shape, lab.dtype)


### 语音是怎么变成「一行一个 429 维向量」的

原始 TIMIT 是 `16 kHz` 的波形（16 秒的句子 ≈ 25.6 万个采样点）。变成下面这一行 429 维，中间经过四步：

| 步骤 | 做了什么 | 关键数字 |
|---|---|---|
| ① 采样 | 波形数字化 | 16,000 点/秒 |
| ② 分帧 | 用滑动窗切片段 | **窗长 25 ms（400 点）、步长 10 ms（160 点）** |
| ③ 特征 | 每帧算 MFCC | 13 维倒谱（含能量）+ 13 维一阶差分 + 13 维二阶差分 = **39 维** |
| ④ 拼上下文 | 前后各取 5 帧拼起来 | 11 帧 × 39 维 = **429 维** |

所以：

> **这 429 维不是「一帧的特征」，而是「以某一帧为中心、前后各 5 帧、共 11 帧的特征」。**
> 每一行对应的标签，是**中心那一帧**的音素。

而**行与行之间的步长仍然是 10 ms**，这带来一个非常重要、但很容易忽略的后果：

```
第 i   行 = 帧 [t-5, ..., t, ..., t+5]
第 i+1 行 = 帧 [t-4, ..., t+1, ..., t+6]
                    └────── 共享 10 帧 ──────┘
```

**相邻两行有 10/11（≈ 91%）的内容是重复的。** 这一点决定了后面切分方式怎么选，也解释了为什么再往上叠窗口收益会迅速衰减（有效上下文已经不短了：首尾跨度 11 帧 ≈ 125 ms）。

顺便可以算一下数据规模：1,229,932 帧 ÷ 100 帧/秒 ≈ **3.4 小时语音**；测试集 451,552 帧 ≈ 1.25 小时。


In [ ]:
# 用 1 秒钟验证上面两个论断：①行内 11 块是连续帧 ②相邻行高度重叠
s = np.asarray(train[:2000], dtype=np.float32)          # (2000, 429)
prev = np.asarray(train[1:2001], dtype=np.float32)

row_corr = np.corrcoef(s.ravel(), prev.ravel())[0, 1]
blocks = [s[:, j*39:(j+1)*39] for j in range(11)]
adj = [np.corrcoef(blocks[j].ravel(), blocks[j+1].ravel())[0, 1] for j in range(10)]
far = np.corrcoef(blocks[0].ravel(), blocks[10].ravel())[0, 1]

print(f'相邻两行（步长 10 ms）的相关系数 : {row_corr:.3f}   <- 接近 1，说明高度重叠')
print(f'行内相邻 39 维块的相关性        : {np.round(adj, 3)}')
print(f'行内第 0 块 vs 第 10 块的相关性  : {far:.3f}   <- 接近 0，说明两端确实差得远')
print()
print('结论：一行 = 11 帧 × 39 维；相邻行 = 滑动 1 帧的同一个窗口')


## Create Dataset

> 官方这里是固定 429 维的 `TIMITDataset` + 80/20 切分；这里换成会拼上下文的 `WindowDataset`，并改用 100% 数据（理由见上一节和文末）。

In [ ]:
K = 2                       # 上下文：t-K .. t+K
IN_DIM = 429 * (2*K + 1)    # 2145
BATCH = 512
NUM_EPOCH = 30
LR = 1.5e-3
SEED = 11

class WindowDataset(Dataset):
    """Returns (2K+1)*429 floats for frame i.

    拼接放在 __getitem__ 里做：预生成要 1.23M x 2145 x 4B = 10.5 GB，会爆内存。
    """
    def __init__(self, X, y=None, k=K, offset=0):
        self.X, self.y, self.k, self.offset = X, y, k, offset
        self.N = X.shape[0]
    def __len__(self):
        return self.N
    def __getitem__(self, i):
        c = i + self.offset
        idx = np.clip(np.arange(c-self.k, c+self.k+1), 0, self.N-1)   # 边界复制
        x = np.asarray(self.X[idx], dtype=np.float32).reshape(-1)
        if self.y is None:
            return torch.from_numpy(x)
        return torch.from_numpy(x), int(self.y[i])

train_loader = DataLoader(WindowDataset(train, lab), batch_size=BATCH,
                          shuffle=True, num_workers=4, pin_memory=True, drop_last=True)
print('in_dim', IN_DIM, '| steps/epoch', len(train_loader))


### 「切分」在这份数据里有两层含义

这两件事经常被混成一件，但它们解决的是完全不同的问题：

### 先看清楚：K=2 到底叠了什么

![K=2 的 5 行拼接示意](../docs/images/hw02-context-k2.png)

五个横条是依次取的 5 行（i-2 … i+2），每行自己已经是 11 帧、125 ms。
它们错开 1 帧，并集 15 帧 = 165 ms；但**中间帧在 5 个横条里都出现**，
所以 2145 维里越靠近中心的帧权重越高——实际新增信息只有左右各 4 帧。

### 第一层：帧内切分（一个样本看多宽）

就是上面的 `K`：取 `t-K..t+K` 共 `2K+1` 行拼成 2145 维。

| K | 输入维度 | 有效跨度（含数据自带的 11 帧） | 说明 |
|---|---|---|---|
| 0 | 429 | 125 ms | 只用数据自带上下文 |
| 1 | 1287 | 约 145 ms | 与 K=0 差别很小 |
| **2** | **2145** | **约 165 ms** | 本项目采用，比 K=1/K=3 都好 |
| 3 | 3003 | 约 185 ms | 收益转负（过平滑） |

### 第二层：数据集切分（能不能信验证分数）

早期版本（v3）用「前 80% 训练、后 20% 验证」。这里**不能改成随机切**，原因是第一层：

- 相邻两行共享 10/11 内容（上面验证过，相关系数接近 1）
- 随机切分时，验证集里某一行，它的「几乎相同」的邻居大概率落在训练集里
- 模型只要记住那个邻居，就能在验证集上拿到虚高的分数

**我们实测过这件事**：用一个线性模型（容量极小、无法记忆）在 30 万帧上对比两种切法，结果是 0.5216（顺序切）vs 0.5231（随机切）——几乎没差。这说明「泄漏」不是自动发生的，它需要模型有记忆能力才会体现。但对 v3 那种 280 万参数的多层 MLP，风险是真实存在的，所以保守起见始终用顺序切。

> 这也是判断「随机切安不安全」的通用方法：**做对照实验，而不是凭直觉下结论。**

### 三个实现细节

1. **边界怎么处理**：`np.clip` 把索引夹在 `[0, N-1]`，即边界处复制首/尾帧（等价于 padding 成 edge）。补零也可行，差别很小。
2. **为什么按需拼而不预生成**：预生成要 `1.23M × 2145 × 4B ≈ 10.5 GB`，Kaggle 内存放不下；按需拼每条样本只需 8.5 KB。
3. **为什么不直接用 RNN / Transformer 跑序列**：`.npy` 是把所有语音首尾拼接的扁平数组，没有 utterance 边界，RNN 会把上一句的隐状态带进下一句。**但这不等于「不能用」**——可以切 200 帧一块做 chunk 训练，代价是少数块切在句中、产生脏样本。

   真正的理由是性价比：k=1(145ms) → k=2(165ms) 有小涨，k=3(185ms) 就掉分，说明这题需要的信息是局部的；而注意力是 O(T²)、123 万帧必须切块，成本翻几倍却换不到长程收益。要上序列模型，**1D 卷积 / TDNN 比 Transformer 划算得多**（文献里 TIMIT 帧级分类用 TDNN/BLSTM 能到 0.78–0.80）。

> 最后一点也解释了为什么这题的分数天花板大概在 0.78–0.80：相邻音素的协同发音会让**边界帧**的标签本身就有歧义，而标签来自强制对齐（forced alignment），边界处本来就没有唯一正确答案。

---

### 切分方式速查

| 层面 | 做法 | 作用 | 踩错会怎样 |
|---|---|---|---|
| 帧内 | `t-K..t+K` 拼接 | 提供上下文 | K 太大 → 过平滑；太小 → 看不到协同发音 |
| 数据集 | 顺序切 80/20 | 得到可信的 val | 随机切 → val 虚高，调参方向全错 |
| 最终训练 | 100%（不切） | 多用 25 万帧 | 无监控，靠已确定的 epoch 数兜底 |


## Create Model

> 官方是 `Linear+Sigmoid` 三层；这里是 `Linear+BatchNorm+ReLU+Dropout`，输入维度从 429 变成 2145。

In [ ]:
class Classifier(nn.Module):
    def __init__(self, in_dim=IN_DIM, h=1024, drop=0.2):
        super().__init__()
        def block(i, o):
            return nn.Sequential(nn.Linear(i, o), nn.BatchNorm1d(o), nn.ReLU(), nn.Dropout(drop))
        self.net = nn.Sequential(block(in_dim, h), block(h, h//2), block(h//2, h//8), nn.Linear(h//8, 39))
    def forward(self, x):
        return self.net(x)


## Training

> 官方用 Adam lr=1e-4、固定 20 epoch；这里是 AdamW 1.5e-3 + OneCycle、30 epoch，并加入 label smoothing。

In [ ]:
def same_seeds(s):
    torch.manual_seed(s); np.random.seed(s); random.seed(s)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

same_seeds(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = Classifier().to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, total_steps=NUM_EPOCH*len(train_loader), pct_start=0.15)
print('params', sum(p.numel() for p in model.parameters()))

for epoch in range(NUM_EPOCH):
    model.train(); ok = n = 0; t0 = time.time()
    for xb, yb in train_loader:
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward(); optimizer.step(); scheduler.step()
        ok += (out.argmax(1) == yb).sum().item(); n += len(yb)
    print(f'[{epoch+1:03d}/{NUM_EPOCH}] train acc {ok/n:.5f} ({time.time()-t0:.0f}s)', flush=True)

torch.save(model.state_dict(), 'model.ckpt')
print('没有验证集，所以只能看 train acc —— 这一点在 README 里解释过')


## Testing

> 官方只做一次前向；这里加了 TTA（窗口偏移 -1/0/+1 取平均），并保存 `probs.npy` 供集成用。

In [ ]:
model.eval()
probs = None
for off in (-1, 0, 1):                       # TTA
    loader = DataLoader(WindowDataset(test, offset=off), batch_size=2048,
                        shuffle=False, num_workers=4, pin_memory=True)
    chunks = []
    with torch.no_grad():
        for xb in loader:
            chunks.append(F.softmax(model(xb.to(device, non_blocking=True)), 1).cpu().numpy())
    acc = np.concatenate(chunks)
    probs = acc if probs is None else probs + acc
    print('offset', off, 'done', flush=True)

probs = (probs / 3).astype(np.float16)
np.save('probs.npy', probs)
pred = probs.argmax(1)
print('pred', pred.shape, '| classes used', np.unique(pred).size, '/ 39')


## Write Prediction

> 与官方一致：写 `Id,Class` 两列。

In [ ]:
import pandas as pd
sub = pd.read_csv(SUB_PATH)
col = sub.columns[-1]
assert len(pred) == len(sub), (len(pred), len(sub))
sub[col] = pred
sub.to_csv('submission.csv', index=False)
print(sub.head()); print('rows', len(sub))
